# Telemetry constrained recovery allocation

CPU entry point for the released allocation comparison. Quick replay verifies published integer tables and recomputes the policy objectives. Full mode rebuilds all integer flows. This notebook is prepared for Google Colab; the reported study was executed in the documented Linux workspace, not a verified Colab runtime.


In [ ]:
from pathlib import Path
import sys, subprocess, os, platform
repo = Path.cwd()/"telemetry-constrained-recovery"
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","v1.0.0","https://github.com/Harits76/telemetry-constrained-recovery.git",str(repo)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",str(repo/"requirements.txt")],check=True)
print(platform.python_version(), platform.platform())


In [ ]:
FULL_FLOW_REBUILD = False
def run(script, *args):
    subprocess.run([sys.executable, str(repo/"src"/script), *args], cwd=repo, check=True)
if not FULL_FLOW_REBUILD:
    run("restore_tables.py")
else:
    for path in (repo/"work").glob("*.bin"):
        path.unlink()
run("run_study.py", "--workers", "2")
run("structure.py")
run("summarize.py")


In [ ]:
run("validate.py")
run("supporting.py")
run("transport_replay.py")
run("communication_audit.py")


In [ ]:
import csv
with (repo/"results/Fullscale_Summary.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(row)


## Packet experiment

The pinned ns-3 build requires a compatible C++23 toolchain, CMake and Ninja. This stage was tested with GCC 14.2 in the Linux workspace; compatibility with the current Colab image must be checked before enabling it. The simulator implements LoRa PHY behavior and the declared query/reply scheduler rather than a complete LoRaWAN MAC.


In [ ]:
RUN_PACKET_EXPERIMENT = False
if RUN_PACKET_EXPERIMENT:
    subprocess.run([sys.executable,"-m","pip","install","-r",str(repo/"ns3/requirements-build.txt")],check=True)
    os.environ["PATH"] = str(Path(sys.executable).parent)+os.pathsep+os.environ["PATH"]
    subprocess.run([sys.executable,str(repo/"ns3/setup.py"),"--jobs","2"],check=True)
    binary=repo/"work/ns-3/build/scratch/ns3.48-telemetry-packets"
    subprocess.run([sys.executable,str(repo/"tests/test_packets.py"),str(binary)],check=True)
    run("packet_experiment.py","prepare")
    subprocess.run([str(binary),"--input="+str(repo/"work/packets/trials.txt"),"--output="+str(repo/"work/packets/Packet_Outputs.csv")],check=True)
    run("packet_experiment.py","analyze")
    run("service_reference.py")


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-r",str(repo/"requirements-figures.txt")],check=True)
run("plot_figures.py")
run("plot_packets.py")
import shutil
archive=shutil.make_archive(str(Path.cwd()/"recovery-results"),"zip",repo,"results")
print(archive)
